# Análisis Hidráulico de Canal de Riego
# Curso: Programación de Computadores con Python
# Ingenierías Civil y Agroindustrial
# Fundamentos de programacion

#

In [ ]:

import math

print("=" * 55)
print("  ANÁLISIS HIDRÁULICO DE CANAL TRAPEZOIDAL DE RIEGO")
print("=" * 55)

  ANÁLISIS HIDRÁULICO DE CANAL TRAPEZOIDAL DE RIEGO


In [ ]:
def calcular_area(b, z, y):
    """Calcula el área transversal de un canal trapezoidal (A = (b + z*y)*y)."""
    return (b + (z * y)) * y

def calcular_perimetro(b, z, y):
    """Calcula el perímetro mojado de un canal trapezoidal (P = b + 2*y*sqrt(1 + z^2))."""
    return b + (2 * y * math.sqrt(1 + z**2))

def calcular_ancho_superficial(b, z, y):
    """Calcula el ancho superior del espejo de agua (T = b + 2*z*y)."""
    return b + (2 * z * y)

In [ ]:
def calcular_caudal(b, z, y, n, S):
    """Calcula el caudal (Q) usando Manning y la velocidad (V) del flujo."""
    if b <= 0 or y <= 0 or n <= 0 or S <= 0 or z < 0:
        print("Error: Los parámetros geométricos e hidráulicos deben ser positivos.")
        return 0.0, 0.0

    A = calcular_area(b, z, y)
    P = calcular_perimetro(b, z, y)

    if P == 0:
        return 0.0, 0.0

    R = A / P  # Radio hidráulico

    # Ecuación de Manning
    Q = (1 / n) * A * (R ** (2/3)) * (math.sqrt(S))
    V = Q / A  # Velocidad

    return Q, V

def clasificar_regimen(Fr):
    """Clasifica el tipo de flujo según el número de Froude."""
    if Fr < 1.0:
        return "Subcrítico"
    elif math.isclose(Fr, 1.0, rel_tol=1e-3):
        return "Crítico"
    else:
        return "Supercrítico ⚠️ Riesgo de erosión"

In [ ]:
def analizar_canal(b, z, n, S, y_min, y_max, paso):
    """Itera en un rango de tirantes y almacena los resultados hidráulicos."""
    lista_resultados = []
    g = 9.81  # Gravedad en m/s²

    y_actual = y_min
    while y_actual <= y_max + 1e-5:  # Margen por tolerancia de flotantes
        A = calcular_area(b, z, y_actual)
        P = calcular_perimetro(b, z, y_actual)
        T = calcular_ancho_superficial(b, z, y_actual)
        Q, V = calcular_caudal(b, z, y_actual, n, S)

        Dh = A / T  # Profundidad hidráulica
        Fr = V / math.sqrt(g * Dh)  # Número de Froude
        regimen = clasificar_regimen(Fr)

        dict_tirante = {
            "y": y_actual, "A": A, "P": P, "R": A/P,
            "Q": Q, "V": V, "Fr": Fr, "regimen": regimen
        }
        lista_resultados.append(dict_tirante)

        y_actual += paso

    return lista_resultados

In [ ]:
def calcular_demanda_cultivo(nombre, ETo, Kc, area_ha):
    """Convierte la demanda hídrica de mm/día en ha a m³/s."""
    area_m2 = area_ha * 10000
    # Conversión considerando mm a metros (dividido entre 1000) y el día a segundos
    Q_demanda = (ETo * Kc * area_m2) / 86400000
    return Q_demanda

def verificar_abastecimiento(Q_canal, cultivos):
    """Evalúa si el caudal disponible cubre la necesidad de cada cultivo."""
    print("\n" + "="*60)
    print(f"VERIFICACIÓN DE ABASTECIMIENTO (Q Canal = {Q_canal:.3f} m³/s)")
    print("="*60)

    for c in cultivos:
        Q_dem = calcular_demanda_cultivo(c["nombre"], c["ETo"], c["Kc"], c["area_ha"])

        if Q_canal >= Q_dem:
            estado = "✅ ABASTECIDO"
        else:
            estado = "❌ NO ABASTECIDO"

        print(f"Cultivo: {c['nombre']:<7} | Demanda: {Q_dem:.3f} m³/s | Estado: {estado}")

In [ ]:
# ── Parámetros del canal ──────────────────────────
b = 0.80
z = 1.5
n = 0.014
S = 0.0008

# ── Análisis hidráulico ───────────────────────────
resultados = analizar_canal(b, z, n, S, 0.20, 1.20, 0.20)

print(f"\n{'y(m)':>6} {'A(m²)':>7} {'P(m)':>7} {'R(m)':>7} {'Q(m³/s)':>9} {'V(m/s)':>7} {'Fr':>7} {'Régimen':>14}")
print("-" * 75)
for r in resultados:
    print(f"{r['y']:>6.2f} {r['A']:>7.3f} {r['P']:>7.3f} {r['R']:>7.3f} {r['Q']:>9.3f} {r['V']:>7.3f} {r['Fr']:>7.3f} {r['regimen']:>14}")

# ── Cultivos ──────────────────────────────────────
cultivos = [
    {"nombre": "Maíz",  "ETo": 5.2, "Kc": 1.05, "area_ha": 40},
    {"nombre": "Arroz", "ETo": 6.0, "Kc": 1.20, "area_ha": 350},
    {"nombre": "Palma", "ETo": 5.5, "Kc": 1.10, "area_ha": 600},
    {"nombre": "Caña",  "ETo": 5.8, "Kc": 1.25, "area_ha": 800},
]

Q_diseño = 0.663   # Caudal para y = 0.80 m
verificar_abastecimiento(Q_diseño, cultivos)


  y(m)   A(m²)    P(m)    R(m)   Q(m³/s)  V(m/s)      Fr        Régimen
---------------------------------------------------------------------------
  0.20   0.220   1.521   0.145     0.122   0.557   0.448     Subcrítico
  0.40   0.560   2.242   0.250     0.449   0.801   0.483     Subcrítico
  0.60   1.020   2.963   0.344     1.012   0.992   0.506     Subcrítico
  0.80   1.600   3.684   0.434     1.854   1.159   0.523     Subcrítico
  1.00   2.300   4.406   0.522     3.013   1.310   0.538     Subcrítico
  1.20   3.120   5.127   0.609     4.527   1.451   0.550     Subcrítico

VERIFICACIÓN DE ABASTECIMIENTO (Q Canal = 0.663 m³/s)
Cultivo: Maíz    | Demanda: 0.025 m³/s | Estado: ✅ ABASTECIDO
Cultivo: Arroz   | Demanda: 0.292 m³/s | Estado: ✅ ABASTECIDO
Cultivo: Palma   | Demanda: 0.420 m³/s | Estado: ✅ ABASTECIDO
Cultivo: Caña    | Demanda: 0.671 m³/s | Estado: ❌ NO ABASTECIDO


In [ ]:
print("\n" + "="*60)
print("  OPTIMIZACIÓN: BUSCANDO TIRANTE MÍNIMO REQUERIDO")
print("="*60)

# 1. Calcular la demanda total combinada
Q_demanda_total = 0
for c in cultivos:
    Q_demanda_total += calcular_demanda_cultivo(c["nombre"], c["ETo"], c["Kc"], c["area_ha"])

print(f"Demanda total requerida para todo el distrito: {Q_demanda_total:.3f} m³/s")

# 2. Búsqueda incremental del tirante óptimo
y_optimo = 0.01
paso_optimo = 0.01
Q_actual = 0.0

while Q_actual < Q_demanda_total:
    y_optimo += paso_optimo
    Q_actual, V_actual = calcular_caudal(b, z, y_optimo, n, S)

print(f"¡Solución encontrada!")
print(f"-> Tirante mínimo necesario (y): {y_optimo:.2f} m")
print(f"-> Caudal generado en el canal: {Q_actual:.3f} m³/s")


  OPTIMIZACIÓN: BUSCANDO TIRANTE MÍNIMO REQUERIDO
Demanda total requerida para todo el distrito: 1.408 m³/s
¡Solución encontrada!
-> Tirante mínimo necesario (y): 0.71 m
-> Caudal generado en el canal: 1.438 m³/s


## Reflexión

Este análisis nos ha permitido comprender cómo los parámetros de un canal de riego, como el tirante de agua, afectan directamente el caudal disponible y, consecuentemente, la capacidad para abastecer las necesidades hídricas de diferentes cultivos. La optimización del tirante es crucial para garantizar un uso eficiente del recurso hídrico, balanceando la oferta del canal con la demanda agrícola.